In [37]:
user = frozendict(name='rahul', age=27)
v: int = 'ss'
name: int = user['name']
age = user['age']
print(name)
print(age)
print(user['age'])

rahul
27
27


# frozendict literal-key inference (basedpyright fork)

Each `reveal_type(...)` shows the checker's result as a note in the editor (and prints the runtime type when run).
Hover any expression to see its type.

## ✅ Works today


In [38]:
from typing import reveal_type

user = frozendict(name='rahul', age=27)

# Each key keeps its own type (stock basedpyright says str | int for both)
reveal_type(user['name'])  # str
reveal_type(user['age'])  # int

# A typo'd key is an error: "Could not access item in frozendict"
try:
    user['nme']
except KeyError as e:
    print('KeyError at runtime too:', e)

# Key completion: type  user[  and the editor offers "name", "age"

# A dict display works too
cfg = frozendict({'lr': 1e-3, 'layers': (64, 64)})
reveal_type(cfg['lr'])  # float
reveal_type(cfg['layers'])  # tuple[Literal[64], Literal[64]]


# The shape survives an inferred return type
def make():
    return frozendict(name='a', age=1)


reveal_type(make()['age'])  # int

KeyError at runtime too: 'nme'


Runtime type is 'str'
Runtime type is 'int'
Runtime type is 'float'
Runtime type is 'tuple'
Runtime type is 'int'


1

## ❌ Doesn't work yet: the next steps

Each line shows what the checker says **today** and what we **want**.


In [39]:
user = frozendict(name='rahul', age=27)

# 1. Hover / printed type doesn't show the shape
reveal_type(user)  # today: frozendict[str, str | int]   want: shows {name: str, age: int}

# 2. .get()
reveal_type(user.get('age'))  # today: str | int | None   want: int
reveal_type(user.get('age', 0))  # today: str | int          want: int
_ = user.get('nme')  # today: no error            want: flagged (or at least None)

# 3. .copy()
reveal_type(user.copy()['age'])  # today: str | int   want: int

# 4. Merging with |
merged = user | {'city': 'delhi'}
reveal_type(merged['age'])  # today: str | int   want: int
reveal_type((user | frozendict(age='x'))['age'])  # today: str | int   want: str (right side wins)

Runtime type is 'frozendict'
Runtime type is 'int'
Runtime type is 'int'
Runtime type is 'int'
Runtime type is 'int'
Runtime type is 'str'


'x'

## Known limitations (not planned yet)


In [40]:
# A declared return type loses the shape (only inferred return types keep it)
def make_user() -> frozendict[str, str | int]:
    return frozendict(name='a', age=1)


reveal_type(make_user()['age'])  # today: str | int   (the annotation can't spell a shape)

# Built from a variable: no shape, just the normal dict typing
d = {'a': 1}
reveal_type(frozendict(d)['a'])  # int via _VT, but a typo key would not be flagged

Runtime type is 'int'
Runtime type is 'int'


1